## Prevendo o gênero de uma música

Esse notebook é uma tentativa de prever o gênero de uma música triangulando os dados do dataset `data.csv` (o principal) onde eu vou pegar a música e o nome dos artistas dessa música, `data_w_genres` onde eu irei extrair a lista completa de gêneros desses artistas.

E por fim, usarei alguns métodos para tentar inferir qual é o gênero da música com base nas possíveis possibilidades.

Inicialmente usarei similaridade de cossenos. Transformando os valores numéricos das caracteristicas da música em um vetor e comparando os vetores
TODO: Depois quero tentar usar regressão linear multivariavel para encontrar o gênero que mais se encaixa com a música também (falando com o gpt, não faz sentido usar pra esse caso)

In [2]:
import pandas as pd
import numpy as np
import ast
import warnings

warnings.filterwarnings('ignore')

def a(x):
    if x.artists:
        x['artists'] = ast.literal_eval(x.artists)
    return x

# Aqui depois eu posso trocar pelo dataset output e tudo deveria funcionar
main_df = pd.read_csv('./spotify-main-dataset/data.csv')
main_df = main_df.apply(a, axis=1)
main_df.head()


,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.0594,1921,0.982,"[Sergei Rachmaninoff, James Levine, Berliner P...",0.279,831667,0.211,0,4BJqT0PrAfrxzMOxytFOIz,0.878000,10,0.665,-20.096,1,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4,1921,0.0366,80.954
1,0.9630,1921,0.732,[Dennis Day],0.819,180533,0.341,0,7xPhfUan2yNtyFG0cUWkt8,0.000000,7,0.160,-12.441,1,Clancy Lowered the Boom,5,1921,0.4150,60.936
2,0.0394,1921,0.961,[KHP Kridhamardawa Karaton Ngayogyakarta Hadin...,0.328,500062,0.166,0,1o6I8BglA6ylDMrIELygv1,0.913000,3,0.101,-14.850,1,Gati Bali,5,1921,0.0339,110.339
3,0.1650,1921,0.967,[Frank Parker],0.275,210000,0.309,0,3ftBPsC5vPBKxYSee08FDH,0.000028,5,0.381,-9.316,1,Danny Boy,3,1921,0.0354,100.109
4,0.2530,1921,0.957,[Phil Regan],0.418,166693,0.193,0,4d6HGyGT8e121BsdKmw9v6,0.000002,3,0.229,-10.096,1,When Irish Eyes Are Smiling,2,1921,0.0380,101.665


In [3]:
genres_df = pd.read_csv('./spotify-main-dataset/data_w_genres.csv')
def b(x):
    if x.genres:
        x['genres'] = ast.literal_eval(x.genres)
    return x

genres_df = genres_df.apply(b, axis=1)
genres_df.head()

,genres,artists,acousticness,danceability,duration_ms,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence,popularity,key,mode,count
0,[show tunes],"""Cats"" 1981 Original London Cast",0.590111,0.467222,250318.555556,0.394003,0.011400,0.290833,-14.448000,0.210389,117.518111,0.389500,38.333333,5,1,9
1,[],"""Cats"" 1983 Broadway Cast",0.862538,0.441731,287280.000000,0.406808,0.081158,0.315215,-10.690000,0.176212,103.044154,0.268865,30.576923,5,1,26
2,[],"""Fiddler On The Roof” Motion Picture Chorus",0.856571,0.348286,328920.000000,0.286571,0.024593,0.325786,-15.230714,0.118514,77.375857,0.354857,34.857143,0,1,7
3,[],"""Fiddler On The Roof” Motion Picture Orchestra",0.884926,0.425074,262890.962963,0.245770,0.073587,0.275481,-15.639370,0.123200,88.667630,0.372030,34.851852,0,1,27
4,[],"""Joseph And The Amazing Technicolor Dreamcoat""...",0.510714,0.467143,270436.142857,0.488286,0.009400,0.195000,-10.236714,0.098543,122.835857,0.482286,43.000000,5,1,7


In [4]:
# Vamos escolher uma música agora

music = main_df[main_df["artists"].str.contains("Legião Urbana", regex=False)]
music = music.iloc[1] # Essa música é "Será" do legião urbana
music

valence                               0.42
year                                  1985
acousticness                         0.122
artists                    [Legião Urbana]
danceability                         0.281
duration_ms                         150467
energy                               0.866
explicit                                 0
id                  7hkQhMFq4EOTYwX3I7cgmA
instrumentalness                   0.00355
key                                      0
liveness                             0.324
loudness                           -10.628
mode                                     1
name                                  Será
popularity                              65
release_date                    1985-01-01
speechiness                         0.0647
tempo                              193.928
Name: 68113, dtype: object

In [5]:
def get_music_possible_genres(music): 
    artists = music['artists']
    possible = set()
    
    for artist in artists:
        s = genres_df[genres_df['artists'].str.contains(artist, regex=False)]['genres']
        
        # s is a Series of lists → flatten it
        for genre_list in s:
            possible.update(genre_list)  # add each string inside the list
    
    return list(possible)

get_music_possible_genres(music)


['rock nacional brasileiro', 'mpb', 'brazilian rock', 'rock brasiliense']

Agora que nós sabemos todos os gêneros que os artistas da música tocam, nós temos uma lista muito menor de generos para comparar com as músicas e escolher. 

Não tenho certeza se o valor musical de um gênero vai ser a média de todos os seus valores ou algo do tipo.

O que eu sei é que eu vou fazer similaridade de cossenos com todos os gêneros possíveis, e escolher o mais próximo.

Eu vou criar um vetor com todas as caracteristicas numericas (acousticness, danceability e etc.) menos a popularidade, pq eu imagino que isso varie muito mesmo dentro do gênero

In [6]:
genres_data = pd.read_csv('./spotify-main-dataset/data_by_genres.csv')
genres_data.head()
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

features = ['acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'speechiness', 'valence']
g = genres_data[genres_data['genres'] == 'orchestra'][features]
m = music[features].values.reshape(1, -1)
similarity = cosine_similarity(m, g)[0][0]
print('fist cossine', similarity)

combined = np.vstack([m, g])
scaled = scaler.fit_transform(combined)

m_scaled, g_scaled = scaled[0].reshape(1, -1), scaled[1].reshape(1, -1)

similarity = cosine_similarity(m_scaled, g_scaled)[0][0]
print("Cosine similarity:", similarity)


fist cossine 0.4067898680521115
Cosine similarity: -1.0


A primeira tentativa acima não deu muito certo. porque as features não estão na mesma escala. O vetor não está bem dimensionado. 

Pedi ajuda pro GPT e cheguei nesse código abaixo.

Explicação do código:
https://chatgpt.com/share/68c2bbec-14b4-8013-8d9d-490f0a9b57d6

Mas o mais interessante foi usar seno e cosseno pra representar a relação circular das keys (tons)

Então o 0 (Dó) está bem proximo do 11 (Si), e a maior distancia dele é pro 6 (F#). Isso faz sentido na escala cromática

In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# --- FEATURES ---
features = [
    'acousticness','danceability','energy','instrumentalness',
    'liveness','loudness','speechiness','valence','tempo','key','mode'
]

# --- MAKE SAFE COPIES ---
music_test = music.copy()
genres_test = genres_data.copy()

# Ensure numeric types
for c in features:
    music_test[c] = pd.to_numeric(music_test[c], errors='coerce')
    genres_test[c] = pd.to_numeric(genres_test[c], errors='coerce')

# Encode circular 'key'
def encode_key(df):
    radians = 2 * np.pi * df['key'] / 12
    df['key_sin'] = np.sin(radians)
    df['key_cos'] = np.cos(radians)
    return df.drop(columns=['key'])

music_test = encode_key(music_test)
genres_test = encode_key(genres_test)

# Updated features list
features_numeric = [
    'acousticness','danceability','energy','instrumentalness',
    'liveness','loudness','speechiness','valence','tempo','mode',
    'key_sin','key_cos'
]

# --- SCALE ON FULL DATASET ---
scaler = StandardScaler()
scaler.fit(genres_test[features_numeric].values)

m_vec = scaler.transform(music_test[features_numeric].values.reshape(1, -1))

def similarity_to_genre(genre_name):
    g_all = genres_test[genres_test['genres'] == genre_name][features_numeric]
    g_centroid = scaler.transform(g_all.mean(axis=0).values.reshape(1, -1))
    sim_centroid = cosine_similarity(m_vec, g_centroid)[0][0]
    sims = cosine_similarity(m_vec, scaler.transform(g_all.values)).ravel()
    return sim_centroid, sims.min(), sims.max(), np.median(sims)

# --- COMPARISONS ---
results = []
for genre in ["8-bit", "orchestra", 'classical', 'rock nacional brasileiro', 'rock brasiliense', 'brazilian rock', 'mpb']:
    sim_centroid, smin, smax, smed = similarity_to_genre(genre)
    results.append({
        "genre": genre,
        "similarity_to_centroid": sim_centroid,
        "min_similarity": smin,
        "max_similarity": smax,
        "median_similarity": smed
    })

# Display nicely
results_df = pd.DataFrame(results)
print(results_df)


                      genre  similarity_to_centroid  min_similarity  \
0                     8-bit               -0.134519       -0.134519   
1                 orchestra               -0.288975       -0.288975   
2                 classical               -0.319665       -0.319665   
3  rock nacional brasileiro                0.279811        0.279811   
4          rock brasiliense                0.058273        0.058273   
5            brazilian rock                0.166021        0.166021   
6                       mpb               -0.147608       -0.147608   

   max_similarity  median_similarity  
0       -0.134519          -0.134519  
1       -0.288975          -0.288975  
2       -0.319665          -0.319665  
3        0.279811           0.279811  
4        0.058273           0.058273  
5        0.166021           0.166021  
6       -0.147608          -0.147608  


Agora o código quebrado em funções


In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. Encode circular 'key' ---
def encode_key(df):
    radians = 2 * np.pi * df['key'] / 12
    df = df.copy()
    df['key_sin'] = np.sin(radians)
    df['key_cos'] = np.cos(radians)
    return df.drop(columns=['key'])

# --- 2. Precompute pipeline ---
def prepare_genres_pipeline(genres_df, features=None):
    """
    Preprocess genre dataset:
    - Encodes 'key'
    - Scales features
    - Computes centroids per genre

    Returns:
    - scaler: fitted StandardScaler
    - genre_centroids: pd.DataFrame of centroids
    - genres_scaled: scaled genre dataset (needed for min/max/median)
    - features_numeric: list of numeric feature names
    """
    if features is None:
        features = [
            'acousticness','danceability','energy','instrumentalness',
            'liveness','loudness','speechiness','valence','tempo','key','mode'
        ]

    df = genres_df.copy()
    for c in features:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    df = encode_key(df)

    features_numeric = [
        'acousticness','danceability','energy','instrumentalness',
        'liveness','loudness','speechiness','valence','tempo','mode',
        'key_sin','key_cos'
    ]

    scaler = StandardScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df[features_numeric]), columns=features_numeric)
    df_scaled['genres'] = df['genres'].values

    genre_centroids = df_scaled.groupby('genres')[features_numeric].mean()

    return scaler, genre_centroids, df_scaled, features_numeric

# --- 3. Compute similarity to a specific genre ---
def similarity_to_genre(music_row, genre_name, scaler, genre_centroids, genres_scaled, features_numeric):
    """
    Compute similarity metrics for a single music track to a specific genre.
    """
    # Ensure music_row is a DataFrame with proper columns
    if isinstance(music_row, pd.Series):
        music_df = pd.DataFrame([music_row])
    else:
        music_df = music_row.copy()
    # music_df = music_row.copy()    
    # Encode key
    music_df = encode_key(music_df)
    
    # Scale features
    m_vec = scaler.transform(music_df[features_numeric].values.reshape(1, -1))
    
    # Filter genre tracks
    g_all = genres_scaled[genres_scaled['genres'] == genre_name][features_numeric]
    if g_all.empty:
        raise ValueError(f"No tracks found for genre '{genre_name}'.")

    # Centroid similarity
    g_centroid = genre_centroids.loc[genre_name].values.reshape(1, -1)
    sim_centroid = cosine_similarity(m_vec, g_centroid)[0][0]

    # Similarity to individual tracks
    sims = cosine_similarity(m_vec, g_all.values).ravel()

    return {
        "genre": genre_name,
        "similarity_to_centroid": sim_centroid,
        "min_similarity": sims.min(),
        "max_similarity": sims.max(),
        "median_similarity": np.median(sims)
    }


In [9]:
# 1. Prepare pipeline (run once)
scaler, genre_centroids, genres_scaled, features_numeric = prepare_genres_pipeline(genres_data)

# 2. Compute similarity to a single genre
music_row = music
result = similarity_to_genre(music_row, "orchestra", scaler, genre_centroids, genres_scaled, features_numeric)
print(result)

# 3. Compute similarity for multiple genres
genres_to_check = ["8-bit", "orchestra", "classical"]
results = [similarity_to_genre(music_row, g, scaler, genre_centroids, genres_scaled, features_numeric) for g in genres_to_check]
results_df = pd.DataFrame(results)
print(results_df)
    

{'genre': 'orchestra', 'similarity_to_centroid': np.float64(-0.28897540729440463), 'min_similarity': np.float64(-0.28897540729440463), 'max_similarity': np.float64(-0.28897540729440463), 'median_similarity': np.float64(-0.28897540729440463)}
       genre  similarity_to_centroid  min_similarity  max_similarity  \
0      8-bit               -0.134519       -0.134519       -0.134519   
1  orchestra               -0.288975       -0.288975       -0.288975   
2  classical               -0.319665       -0.319665       -0.319665   

   median_similarity  
0          -0.134519  
1          -0.288975  
2          -0.319665  


In [10]:
def get_music_genre(music, g, scaler, genre_centroids, genres_scaled, features_numeric, n: int = 1):
    possibilities = get_music_possible_genres(music)
    if len(possibilities) == 0:
        return ''
    results = [similarity_to_genre(music_row, g, scaler, genre_centroids, genres_scaled, features_numeric) for g in possibilities]
    results_df = pd.DataFrame(results)
    return results_df.sort_values(by="similarity_to_centroid", ascending=False).iloc[:n].genre
    

Agora que todas as funções estão definidas, ou seja, tenho a `get_music_genre` e a `prepare_genres_pipeline` pra rodar. Posso pegar uma musiquinha do Alice in Chains pra achar o gênero dela

In [11]:
m = main_df[main_df['artists'].str.contains('Alice In Chains', regex=False)]
m

,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
13608,0.758,1990,0.000450,[Alice In Chains],0.355,285200,0.791,1,6gZVQvQZOFpzIy3HblJ20F,0.000000,8,0.0969,-7.565,1,Man in the Box,73,1990-08-01,0.0453,106.392
13700,0.525,1990,0.000018,[Alice In Chains],0.486,152560,0.938,0,5Ds35L9KpUDKgSxZ6whuoQ,0.000103,1,0.3610,-6.652,1,We Die Young,58,1990-08-01,0.0704,125.816
13736,0.221,1990,0.002520,[Alice In Chains],0.245,242600,0.749,0,60l2m3BD5VY0HSc3xmSpPI,0.000008,8,0.1850,-9.352,0,Bleed The Freak,54,1990-08-01,0.0599,118.363
13789,0.311,1990,0.013900,[Alice In Chains],0.288,349747,0.843,0,4XssnBT81vTIH6iYYUSv84,0.000000,11,0.0555,-7.858,1,Sea Of Sorrow,51,1990-08-01,0.0633,122.147
13799,0.209,1990,0.007830,[Alice In Chains],0.323,387907,0.633,0,2YolSRrYvFxhTGjbiT33yH,0.000112,8,0.0435,-9.014,1,"Love, Hate, Love",52,1990-08-01,0.0402,103.978
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119986,0.372,1995,0.000179,[Alice In Chains],0.455,165733,0.973,0,59tm0GSryH6BozhjGOergk,0.083000,4,0.1090,-5.278,1,So Close,39,1995-10-30,0.0561,142.497
134896,0.434,1990,0.014000,[Alice In Chains],0.358,337827,0.819,0,48zAaZoXJxURbEHzxDDHXy,0.000141,6,0.1240,-5.495,1,Down in a Hole,31,1990,0.0328,97.197
135316,0.571,1992,0.000705,[Alice In Chains],0.300,206827,0.814,0,2da0KrvLB5acEfw9bCnwQY,0.000005,6,0.1210,-10.584,1,Would?,34,1992-06-02,0.0375,100.489
153498,0.113,2009,0.822000,[Alice In Chains],0.205,183560,0.331,0,78iMIaSjeVlUoNa7rehPOU,0.000695,8,0.0891,-9.265,1,Black Gives Way To Blue,46,2009-01-01,0.0302,179.982


In [21]:

get_music_genre(m.iloc[0], g, scaler, genre_centroids, genres_scaled, features_numeric, 2)

5    alternative rock
2              grunge
Name: genre, dtype: object

Agora só um teste manual pra ver que os resultados batem

In [13]:

music_row = m.iloc[0]
print(music_row)

genres_to_check = ['alternative metal',
 'hard rock',
 'alternative rock',
 'nu metal',
 'grunge',
 'rock']
results = [similarity_to_genre(music_row, g, scaler, genre_centroids, genres_scaled, features_numeric) for g in genres_to_check]
results_df = pd.DataFrame(results)
sort = results_df.sort_values(by="similarity_to_centroid", ascending=False)
print(sort)

valence                              0.758
year                                  1990
acousticness                       0.00045
artists                  [Alice In Chains]
danceability                         0.355
duration_ms                         285200
energy                               0.791
explicit                                 1
id                  6gZVQvQZOFpzIy3HblJ20F
instrumentalness                       0.0
key                                      8
liveness                            0.0969
loudness                            -7.565
mode                                     1
name                        Man in the Box
popularity                              73
release_date                    1990-08-01
speechiness                         0.0453
tempo                              106.392
Name: 13608, dtype: object
               genre  similarity_to_centroid  min_similarity  max_similarity  \
2   alternative rock                0.671076        0.671076        0.671076

Agora nós vamos popular todo o dataset

In [22]:
# OPTIMIZED VERSION FOR 100K+ RECORDS

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import time

def build_artist_genre_cache():
    """
    Build a fast lookup cache for artist -> genres mapping
    This eliminates the need for string matching on every song
    """
    print("Building artist-genre cache...")
    start_time = time.time()
    
    artist_genres = defaultdict(set)
    
    for _, row in genres_df.iterrows():
        artist_name = row['artists']
        genres_list = row['genres'] if isinstance(row['genres'], list) else []
        
        # Clean artist name and add to cache
        if isinstance(artist_name, str):
            artist_genres[artist_name.strip()].update(genres_list)
    
    print(f"Cache built in {time.time() - start_time:.2f}s with {len(artist_genres)} artists")
    return dict(artist_genres)

def get_music_possible_genres_fast(artists_list, artist_genre_cache):
    """
    Fast genre lookup using prebuilt cache
    """
    possible_genres = set()
    
    for artist in artists_list:
        if isinstance(artist, str):
            # Exact match first
            if artist in artist_genre_cache:
                possible_genres.update(artist_genre_cache[artist])
            else:
                # Fallback: partial matching (slower but more accurate)
                for cached_artist, genres in artist_genre_cache.items():
                    if artist.lower() in cached_artist.lower() or cached_artist.lower() in artist.lower():
                        possible_genres.update(genres)
                        break
    
    return list(possible_genres)

def batch_predict_genres(music_batch, artist_genre_cache, scaler, genre_centroids, features_numeric, batch_size=1000):
    """
    Vectorized batch processing for much faster prediction
    """
    results = []
    
    # Preprocess the entire batch at once
    print("Preprocessing batch...")
    batch_df = music_batch.copy()
    
    # Encode keys for the entire batch
    radians = 2 * np.pi * batch_df['key'] / 12
    batch_df['key_sin'] = np.sin(radians)
    batch_df['key_cos'] = np.cos(radians)
    batch_df = batch_df.drop(columns=['key'])
    
    # Scale all features at once (MUCH faster than one-by-one)
    music_features = batch_df[features_numeric].values
    scaled_music = scaler.transform(music_features)
    
    print("Predicting genres for batch...")
    
    for i, (idx, row) in enumerate(batch_df.iterrows()):
        if i % 1000 == 0:
            print(f"Processing {i}/{len(batch_df)}...")
            
        # Fast genre lookup
        possible_genres = get_music_possible_genres_fast(row['artists'], artist_genre_cache)
        
        if not possible_genres:
            results.append('')
            continue
        
        # Vectorized similarity computation for all possible genres
        music_vector = scaled_music[i:i+1]  # Single row but keep 2D shape
        
        best_genre = ''
        best_similarity = -1
        
        for genre in possible_genres:
            if genre in genre_centroids.index:
                genre_centroid = genre_centroids.loc[genre].values.reshape(1, -1)
                similarity = cosine_similarity(music_vector, genre_centroid)[0][0]
                
                if similarity > best_similarity:
                    best_similarity = similarity
                    best_genre = genre
        
        results.append(best_genre)
    
    return results

def get_music_genre_fast(music, g, scaler, genre_centroids, genres_scaled, features_numeric, cache, n: int = 1):
    possibilities = get_music_possible_genres_fast(music['artists'], cache)
    if len(possibilities) == 0:
        return ''
    results = [similarity_to_genre(music_row, g, scaler, genre_centroids, genres_scaled, features_numeric) for g in possibilities]
    results_df = pd.DataFrame(results)
    return results_df.sort_values(by="similarity_to_centroid", ascending=False).iloc[:n].genre
     
# Build the cache once
artist_genre_cache = build_artist_genre_cache()


Building artist-genre cache...
Cache built in 1.11s with 28678 artists


In [30]:
def get_predicted_genre(row):
    result = get_music_genre_fast(row, g, scaler, genre_centroids, genres_scaled, features_numeric, artist_genre_cache, n=1)
    return result.iloc[0] if len(result) > 0 else ''

test_df = main_df.copy()
test_df['predicted_genre'] = test_df.apply(get_predicted_genre, axis=1)

In [32]:
len(test_df['predicted_genre'].unique())
test_df.to_csv('main_dataset_w_genre.csv')
